In [1]:
# # Grid filters: histogram filter and binary Bayes filter (notebook)
#
# The hallway of the Note: 10 cells of 1 m in a loop, doors at cells 1, 3 and 7. Door sensor: p(door | door cell) = 0.6,
# p(door | wall cell) = 0.2. Command "move 2": 1, 2 or 3 cells with probability 0.1, 0.8, 0.1. Then the door world for
# the binary Bayes filter, and the three-step hallway run for smoothing and the Viterbi algorithm.

In [2]:
import itertools

import numpy as np
import plotly.graph_objects as go

N = 10
DOOR = np.zeros(N, bool)
DOOR[[1, 3, 7]] = True
KERNEL = {1: 0.1, 2: 0.8, 3: 0.1}


def likelihood(z):                       # z = 1 'door', 0 'wall'
    p = np.where(DOOR, 0.6, 0.2)
    return p if z else 1 - p


def sense(bel, z):
    u = likelihood(z) * bel
    return u / u.sum(), u


def move(bel, kernel=KERNEL):
    out = np.zeros(N)
    for step, p in kernel.items():
        out += p * np.roll(bel, step)    # np.roll shifts every value `step` cells to the right, wrapping round
    return out

## 1. One sense step and one move step by hand (Note, Section 2)

In [3]:
prior = np.full(N, 0.1)
b1, unnorm = sense(prior, 1)
print("products:", unnorm, " sum:", unnorm.sum())
print("after sensing 'door':", b1)
exact = np.roll(b1, 2)
print("exact move of 2:", exact)
bb2 = move(b1)
print("move with 0.1/0.8/0.1:", bb2)
print("cell 3 by hand:", 0.1 * b1[2] + 0.8 * b1[1] + 0.1 * b1[0])
b2, _ = sense(bb2, 1)
print("after the second 'door':", np.round(b2, 4))

products: [0.02 0.06 0.02 0.06 0.02 0.02 0.02 0.06 0.02 0.02]  sum: 0.32000000000000006
after sensing 'door': [0.0625 0.1875 0.0625 0.1875 0.0625 0.0625 0.0625 0.1875 0.0625 0.0625]
exact move of 2: [0.0625 0.0625 0.0625 0.1875 0.0625 0.1875 0.0625 0.0625 0.0625 0.1875]
move with 0.1/0.8/0.1: [0.075  0.0625 0.075  0.1625 0.0875 0.1625 0.075  0.0625 0.075  0.1625]
cell 3 by hand: 0.16249999999999998
after the second 'door': [0.0476 0.119  0.0476 0.3095 0.0556 0.1032 0.0476 0.119  0.0476 0.1032]


## 2. Moving without sensing: the belief flattens (Note, Section 2.4)

In [4]:
bel = np.zeros(N)
bel[1] = 1.0
for k in range(1, 41):
    bel = move(bel)
    if k in (1, 2, 5, 10, 20, 40):
        print(f"after {k:2d} moves: max {bel.max():.4f}, min {bel.min():.4f}")

after  1 moves: max 0.8000, min 0.0000
after  2 moves: max 0.6600, min 0.0000
after  5 moves: max 0.4325, min 0.0000
after 10 moves: max 0.2931, min 0.0017
after 20 moves: max 0.2025, min 0.0180
after 40 moves: max 0.1426, min 0.0584


## 3. A longer run with the true robot (Note, Section 2.5)

In [5]:
true_cells = [1]
for _ in range(6):
    true_cells.append((true_cells[-1] + 2) % N)
readings = [int(DOOR[c]) for c in true_cells]
readings[4] = 1                          # one wrong reading: the sensor says 'door' at the wall of cell 9
bel, _ = sense(np.full(N, 0.1), readings[0])
print("t=1", true_cells[0], readings[0], np.round(bel, 3))
for t in range(1, len(readings)):
    bel, _ = sense(move(bel), readings[t])
    print(f"t={t + 1}", true_cells[t], readings[t], "argmax", bel.argmax(), "max", round(bel.max(), 3))

t=1 1 1 [0.062 0.187 0.062 0.187 0.062 0.062 0.062 0.187 0.062 0.062]
t=2 3 1 argmax 3 max 0.31
t=3 5 0 argmax 5 max 0.302
t=4 7 1 argmax 7 max 0.421
t=5 9 1 argmax 3 max 0.315
t=6 1 1 argmax 1 max 0.372
t=7 3 1 argmax 3 max 0.469


## 4. How many cells? (Note, Section 3)

In [6]:
print("30 m x 30 m floor, 15 cm cells, 2 degrees:", round(30 / 0.15) ** 2 * 180)
print("same floor, 5 cm cells, 1 degree:", round(30 / 0.05) ** 2 * 360)

30 m x 30 m floor, 15 cm cells, 2 degrees: 7200000
same floor, 5 cm cells, 1 degree: 129600000


## 5. Binary Bayes filter on a static door, in log odds (Note, Section 4)

In [7]:
def logit(p):
    return np.log(p / (1 - p))


def prob(l):
    return 1 - 1 / (1 + np.exp(l))


p_open_given = {"open": 0.6 * 0.5 / (0.6 * 0.5 + 0.2 * 0.5),        # inverse model with prior 0.5
                "closed": 0.4 * 0.5 / (0.4 * 0.5 + 0.8 * 0.5)}
print("inverse model:", p_open_given, {k: round(logit(v), 4) for k, v in p_open_given.items()})
l0 = logit(0.5)
l = l0
for z in ["open", "open", "closed", "open", "open"]:
    l = l + logit(p_open_given[z]) - l0
    print(f"after '{z}': log odds {l:.4f}, p(open) {prob(l):.4f}")
# the same numbers straight from Bayes' theorem
p = 0.5
for z in ["open", "open", "closed", "open", "open"]:
    pz_o, pz_c = (0.6, 0.2) if z == "open" else (0.4, 0.8)
    p = pz_o * p / (pz_o * p + pz_c * (1 - p))
print("Bayes' theorem gives", round(p, 4))

inverse model: {'open': 0.7499999999999999, 'closed': 0.3333333333333333} {'open': np.float64(1.0986), 'closed': np.float64(-0.6931)}
after 'open': log odds 1.0986, p(open) 0.7500
after 'open': log odds 2.1972, p(open) 0.9000
after 'closed': log odds 1.5041, p(open) 0.8182
after 'open': log odds 2.6027, p(open) 0.9310
after 'open': log odds 3.7013, p(open) 0.9759
Bayes' theorem gives 0.9759


Why logs: after 40 readings of 'open', the probability rounds to exactly 1.0 and a 'closed' cannot move it.

In [8]:
p = 0.5
for _ in range(40):
    p = 0.6 * p / (0.6 * p + 0.2 * (1 - p))
print("p after 40 'open':", p, " p == 1.0:", p == 1.0)
p_after_closed = 0.4 * p / (0.4 * p + 0.8 * (1 - p))
print("then one 'closed':", p_after_closed)
l = 40 * logit(0.75)
print("log odds after 40 'open':", round(l, 3), " then one 'closed':", round(l + logit(1 / 3), 3))

p after 40 'open': 1.0  p == 1.0: True
then one 'closed': 1.0
log odds after 40 'open': 43.944  then one 'closed': 43.251


A prior other than 0.5: the inverse model already contains the prior once, so the update subtracts l0.

In [9]:
prior_open = 0.2
inv = 0.6 * prior_open / (0.6 * prior_open + 0.2 * (1 - prior_open))
l0 = logit(prior_open)
right = l0 + logit(inv) - l0
wrong = l0 + logit(inv)
print("inverse model p(open | sense open):", round(inv, 4), " l0:", round(l0, 4))
print("with -l0:", round(prob(right), 4), " without:", round(prob(wrong), 4))

inverse model p(open | sense open): 0.4286  l0: -1.3863
with -l0: 0.4286  without: 0.1579


## 6. Filtering, smoothing and the most likely path (Note, Section 5)

Run: t=1 stay, reads door; t=2 move 2, reads door; t=3 move 2, reads wall. True cells 1, 3, 5.

In [10]:
T = np.zeros((N, N))
for i in range(N):
    for step, p in KERNEL.items():
        T[i, (i + step) % N] += p
zs = [1, 1, 0]
alpha = [0.1 * likelihood(zs[0])]
for z in zs[1:]:
    alpha.append(likelihood(z) * (alpha[-1] @ T))
beta = [np.ones(N)]
for z in zs[:0:-1]:
    beta.insert(0, T @ (likelihood(z) * beta[0]))
for t in range(3):
    print(f"t={t + 1} alpha", np.round(alpha[t], 4))
    print(f"    beta ", np.round(beta[t], 4))
    print("    alpha.beta sums to", round((alpha[t] * beta[t]).sum(), 5))
filtered = [a / a.sum() for a in alpha]
smoothed = [a * b / (a * b).sum() for a, b in zip(alpha, beta)]
print("filtered t=1:", np.round(filtered[0], 4))
print("smoothed t=1:", np.round(smoothed[0], 4))
print("smoothed t=2:", np.round(smoothed[1], 4))

t=1 alpha [0.02 0.06 0.02 0.06 0.02 0.02 0.02 0.06 0.02 0.02]
    beta  [0.1984 0.4144 0.1792 0.1072 0.1792 0.4144 0.1792 0.1064 0.1536 0.26  ]
    alpha.beta sums to 0.06896
t=2 alpha [0.0048 0.012  0.0048 0.0312 0.0056 0.0104 0.0048 0.012  0.0048 0.0104]
    beta  [0.72 0.48 0.76 0.8  0.76 0.48 0.76 0.8  0.76 0.48]
    alpha.beta sums to 0.06896
t=3 alpha [0.0049 0.0037 0.0049 0.0042 0.0065 0.0208 0.0069 0.0037 0.0049 0.0084]
    beta  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
    alpha.beta sums to 0.06896
filtered t=1: [0.0625 0.1875 0.0625 0.1875 0.0625 0.0625 0.0625 0.1875 0.0625 0.0625]
smoothed t=1: [0.0575 0.3606 0.052  0.0933 0.052  0.1202 0.052  0.0926 0.0445 0.0754]
smoothed t=2: [0.0501 0.0835 0.0529 0.3619 0.0617 0.0724 0.0529 0.1392 0.0529 0.0724]


In [11]:
m = [0.1 * likelihood(zs[0])]
back = []
for z in zs[1:]:
    cand = m[-1][:, None] * T            # cand[i, j]: best path to i, then a step from i to j
    back.append(cand.argmax(axis=0))
    m.append(likelihood(z) * cand.max(axis=0))
for t, row in enumerate(m):
    print(f"t={t + 1} best-path probability", np.round(row, 5))
path = [int(m[-1].argmax())]
for bp in back[::-1]:
    path.insert(0, int(bp[path[0]]))
print("Viterbi path:", path, " probability", round(m[-1].max(), 6))


def path_prob(p):
    out = 0.1 * likelihood(zs[0])[p[0]]
    for t in range(1, 3):
        out *= T[p[t - 1], p[t]] * likelihood(zs[t])[p[t]]
    return out


best = max(itertools.product(range(N), repeat=3), key=path_prob)
print("brute force over all 1000 paths:", best, round(path_prob(best), 6))

t=1 best-path probability [0.02 0.06 0.02 0.06 0.02 0.02 0.02 0.06 0.02 0.02]
t=2 best-path probability [0.0032 0.0096 0.0032 0.0288 0.0032 0.0096 0.0032 0.0096 0.0032 0.0096]
t=3 best-path probability [0.00205 0.00307 0.00205 0.00307 0.0023  0.01843 0.0023  0.00307 0.00205
 0.00614]
Viterbi path: [1, 3, 5]  probability 0.018432
brute force over all 1000 paths: (1, 3, 5) 0.018432


In [12]:
fig = go.Figure([go.Bar(x=list(range(N)), y=filtered[0], name="filtered, t = 1"),
                 go.Bar(x=list(range(N)), y=smoothed[0], name="smoothed, t = 1")])
fig.update_layout(barmode="group", template="simple_white", xaxis_title="cell", yaxis_title="probability")
fig.show()